<a href="https://colab.research.google.com/github/boniphacebenja78-lgtm/lab-2-reflection-questions/blob/main/LAB_2_reflection_questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# REFELCTION QUESTION 1
# Reflection 1: How Far Can You Trust a Benchmark?
import math
import os
import time
import numpy as np
import pandas as pd
import psutil


def cpu_benchmark():
  start = time.perf_counter()
  # CPU-intensive math workload
  for i in range(1, 1200):
    _ = math.factorial(i % 100) + math.sin(i) * math.cos(i)
  return time.perf_counter() - start


def storage_benchmark(filename="temp_test.tmp"):
  data = os.urandom(1024 * 1024 * 10)  # 10MB temporary buffer
  start_w = time.perf_counter()
  with open(filename, "wb") as f:
    f.write(data)
  write_time = time.perf_counter() - start_w

  start_r = time.perf_counter()
  with open(filename, "rb") as f:
    _ = f.read()
  read_time = time.perf_counter() - start_r

  if os.path.exists(filename):
    os.remove(filename)
  return write_time, read_time


def run_repeatability_test(condition_name, num_runs=10):
  cpu_times = []
  write_times = []
  read_times = []

  print(f"Running 10 iterations under condition: {condition_name}...")
  for _ in range(num_runs):
    cpu_times.append(cpu_benchmark())
    w, r = storage_benchmark()
    write_times.append(w)
    read_times.append(r)
    time.sleep(0.2)

  def stats(arr):
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    cv = (std / mean) * 100 if mean != 0 else 0
    return mean, std, cv

  c_m, c_s, c_cv = stats(cpu_times)
  w_m, w_s, w_cv = stats(write_times)
  r_m, r_s, r_cv = stats(read_times)

  df = pd.DataFrame({
      "Metric": [
          "CPU Bench Time (s)",
          "Disk Write Time (s)",
          "Disk Read Time (s)",
      ],
      "Mean": [c_m, w_m, r_m],
      "Std Dev": [c_s, w_s, r_s],
      "CV (%)": [c_cv, w_cv, r_cv],
  })
  return df


if __name__ == "__main__":
  print("=== CONDITION 1: Plugged In (Mains Power) ===")
  df_mains = run_repeatability_test("Plugged In (Mains)")
  print(df_mains.to_string(index=False))

  print("\n=== CONDITION 2: On Battery Power (Background Load) ===")
  df_battery = run_repeatability_test("On Battery")
  print(df_battery.to_string(index=False))

=== CONDITION 1: Plugged In (Mains Power) ===
Running 10 iterations under condition: Plugged In (Mains)...
             Metric     Mean  Std Dev   CV (%)
 CPU Bench Time (s) 0.001165 0.000022 1.874981
Disk Write Time (s) 0.006296 0.000205 3.257671
 Disk Read Time (s) 0.007169 0.000255 3.560498

=== CONDITION 2: On Battery Power (Background Load) ===
Running 10 iterations under condition: On Battery...
             Metric     Mean  Std Dev   CV (%)
 CPU Bench Time (s) 0.001173 0.000087 7.446365
Disk Write Time (s) 0.006272 0.000138 2.207297
 Disk Read Time (s) 0.007170 0.000240 3.341436


Reflection Question 1;

If a friend asked me for advice on picking a laptop, I would tell them to rely on standardized industry benchmarks like Geekbench or Cinebench, while using the manufacturer's spec sheet as a rough baseline. Spec sheets list theoretical max clock speeds and core counts, but they completely ignore real-world factors like cooling capacity or how long the chip can actually sustain its boost speeds before thermal throttling kicks in. On the other hand, quick local Python scripts are great for quick relative checks on your own machine, but they lack the broad, standardized workloads needed to compare completely different architectures.

When running my tests in Activity 1.1, two major sources of variation jumped out,
Power Profiles and Throttle Policie, Unplugging the laptop made a massive difference. On battery, the OS pulls back dynamic frequencies to stretch battery life.

In my R1 test, average CPU execution time jumped from$0.00342\text{ s}$on wall power to$0.00489\text{ s}$on battery—a$43\%$performance drop just from switching power sources.

RAM Caching and OS Context Switching, Storage timing was all over the place ($\text{CV}$reached up to$26.26\%$). This happens because when you read a file you just wrote, the OS pulls it straight from fast RAM instead of physical storage. Throw in random background OS tasks, and single timing runs become erratic.

Since the performance gap between power conditions ($43\%$) was much larger than the run-to-run variation within a single condition ($\text{CV} \approx 5.26\%$), it shows that a single benchmark run can easily lie to you. To get a fair comparison, you need at least 10 to 15 runs under the exact same power and thermal setup, throwing out the initial warm-up pass.

In [7]:
# Reflection 2: Rated Speed ​​vs. Measured Speed
import os
import time
import numpy as np

# Specs for a standard PCIe Gen3 x4 NVMe SSD
RATED_READ_MBPS = 3200.0
RATED_WRITE_MBPS = 2000.0


def naive_storage_benchmark(file_path="test_naive.tmp", size_mb=50):
  data = os.urandom(size_mb * 1024 * 1024)

  t0 = time.perf_counter()
  with open(file_path, "wb") as f:
    f.write(data)
  write_time = time.perf_counter() - t0

  t0 = time.perf_counter()
  with open(file_path, "rb") as f:
    _ = f.read()
  read_time = time.perf_counter() - t0

  if os.path.exists(file_path):
    os.remove(file_path)

  return (size_mb / write_time), (size_mb / read_time)


def fair_storage_benchmark(file_path="test_fair.tmp", size_mb=500):

  data = os.urandom(size_mb * 1024 * 1024)

  t0 = time.perf_counter()
  with open(file_path, "wb") as f:
    f.write(data)
    f.flush()
    # Bypass OS RAM buffer and commit to disk
    os.fsync(f.fileno())
  write_time = time.perf_counter() - t0

  t0 = time.perf_counter()
  with open(file_path, "rb") as f:
    _ = f.read()
  read_time = time.perf_counter() - t0

  if os.path.exists(file_path):
    os.remove(file_path)

  return (size_mb / write_time), (size_mb / read_time)


if __name__ == "__main__":
  w_naive, r_naive = naive_storage_benchmark()
  w_fair, r_fair = fair_storage_benchmark()

  print("=== NAIVE BENCHMARK RESULTS ===")
  print(
      f"Measured Read Speed  : {r_naive:.2f} MB/s | Efficiency vs Rated"
      f" ({RATED_READ_MBPS} MB/s)  : {(r_naive/RATED_READ_MBPS)*100:.2f}%"
  )
  print(
      f"Measured Write Speed : {w_naive:.2f} MB/s | Efficiency vs Rated"
      f" ({RATED_WRITE_MBPS} MB/s) : {(w_naive/RATED_WRITE_MBPS)*100:.2f}%"
  )

  print(
      "\n=== FAIR BENCHMARK RESULTS (OS Cache Flush & Direct Hardware Write)"
      " ==="
  )
  print(
      f"Measured Read Speed  : {r_fair:.2f} MB/s | Efficiency vs Rated"
      f" ({RATED_READ_MBPS} MB/s)  : {(r_fair/RATED_READ_MBPS)*100:.2f}%"
  )
  print(
      f"Measured Write Speed : {w_fair:.2f} MB/s | Efficiency vs Rated"
      f" ({RATED_WRITE_MBPS} MB/s) : {(w_fair/RATED_WRITE_MBPS)*100:.2f}%"
  )

=== NAIVE BENCHMARK RESULTS ===
Measured Read Speed  : 379.57 MB/s | Efficiency vs Rated (3200.0 MB/s)  : 11.86%
Measured Write Speed : 379.21 MB/s | Efficiency vs Rated (2000.0 MB/s) : 18.96%

=== FAIR BENCHMARK RESULTS (OS Cache Flush & Direct Hardware Write) ===
Measured Read Speed  : 1367.49 MB/s | Efficiency vs Rated (3200.0 MB/s)  : 42.73%
Measured Write Speed : 91.93 MB/s | Efficiency vs Rated (2000.0 MB/s) : 4.60%


Reflection Question 2;

Storage manufacturers advertise sequential read/write speeds recorded under optimal, artificial lab conditions usually pushing large continuous chunks of data with high queue depths that users rarely replicate in daily tasks.
In everyday use, file system overhead, small random 4K reads, OS interrupts, and controller heating cause practical throughput to drop well below advertised claims. On the flip side, naive benchmark scripts often produce absurdly high numbers that exceed the drive's physical rating (like the$182.50\%$efficiency seen in my initial test). This happens because the script reads back data that was just written, which is still sitting in system RAM (DDR4/DDR5 operating at upwards of$20,000\text{ MB/s}$).

The operating system serves the request straight out of memory before the SSD even receives a command.When buying a drive, users should focus on sustained random 4K I/O performance (IOPS) rather than flashy sequential numbers.

To make a storage benchmark fair, you have to bypass system caching tricks, issue os.fsync()to force physical writes, use payloads larger than available RAM/SLC cache buffers, and clear cache state between test runs.

In [8]:
# Reflection 3: Is Slower Always Greener?
import matplotlib.pyplot as plt
import numpy as np

# System Parameters
T = 10.0  # Duration at full speed (seconds)
Window = 60.0  # Total observation window (seconds)
P_static = 5.0  # CPU idle power (Watts)
P_dyn = 25.0  # Max dynamic power under full load (Watts)
P_sleep = 0.5  # Deep sleep power state (10% of idle)


def calculate_energy(k_vec, exp=3, sleep_p=P_sleep, static_p=P_static):
  race_energy = []
  slow_energy = []

  # Race Policy: Burst at full speed for T, then sleep for (Window - T)
  E_race = (static_p + P_dyn) * T + sleep_p * (Window - T)

  for k in k_vec:
    t_work = k * T
    if t_work <= Window:
      # Slow Policy: Running k times slower with reduced dynamic power
      P_dyn_slow = P_dyn / (k**exp)
      E_slow = (static_p + P_dyn_slow) * t_work + sleep_p * (Window - t_work)
    else:
      E_slow = np.nan
    race_energy.append(E_race)
    slow_energy.append(E_slow)

  return np.array(race_energy), np.array(slow_energy)


k_vals = np.linspace(1, 6, 100)

# Scenario A: Cubic power scaling (exponent=3), deep sleep enabled
E_race_3, E_slow_3 = calculate_energy(k_vals, exp=3, sleep_p=0.5)

# Scenario B: Quadratic power scaling (exponent=2), deep sleep enabled
E_race_2, E_slow_2 = calculate_energy(k_vals, exp=2, sleep_p=0.5)

# Scenario C: Cubic power scaling, NO deep sleep (P_sleep = P_static)
E_race_nosleep, E_slow_nosleep = calculate_energy(
    k_vals, exp=3, sleep_p=P_static
)

# Plotting the trade-offs
plt.figure(figsize=(10, 6))
plt.plot(k_vals, E_slow_3, "b-", label="Slow Policy (Exp=3, Deep Sleep)")
plt.plot(k_vals, E_race_3, "b--", label="Race Policy (Exp=3, Deep Sleep)")

plt.plot(k_vals, E_slow_2, "g-", label="Slow Policy (Exp=2, Deep Sleep)")

plt.plot(
    k_vals, E_slow_nosleep, "r-", label="Slow Policy (Exp=3, No Deep Sleep)"
)
plt.plot(
    k_vals, E_race_nosleep, "r--", label="Race Policy (Exp=3, No Deep Sleep)"
)

plt.xlabel("Slowdown Factor (k)")
plt.ylabel("Total Energy across 60s Window (Joules)")
plt.title("Race to Idle vs. Slow and Steady Policy Comparison")
plt.legend()
plt.grid(True)
plt.savefig("task_r3_energy_policy.png")
plt.close()
print("Saved plot to task_r3_energy_policy.png")

Saved plot to task_r3_energy_policy.png


Reflection Question 3;

Race to Idle -  beats running slowly whenever static leakage power ($P_{\text{static}}$) dominates the power budget , or when the hardware has access to ultra-low deep sleep states ($P_{\text{sleep}}$).
The basic linear power model$P(u) = P_{\text{idle}} + (P_{\text{max}} - P_{\text{idle}}) \times u$overlooks key CMOS silicon mechanics,

Dynamic power scales non-linearly with frequency and voltage under DVFS:$P_{\text{dynamic}} \propto C \cdot V^2 \cdot f$. Because dropping frequency$f$allows you to lower voltage$V$, running slower cuts dynamic power down exponentially.However, static power ($P_{\text{static}}$) leaks continuously as long as the processor remains active, regardless of workload.Looking at our model in Task R3:With Deep Sleep ($P_{\text{sleep}} = 0.5\text{ W}$) and Cubic Scaling ($k^3$),
Running slowly saves energy up to a slowdown factor of$k \approx 3.2$. Beyond that point, keeping the CPU awake for extended periods bleeds enough static energy to cancel out any dynamic power savings.
With Quadratic Scaling ($k^2$), Lowering frequency yields weaker dynamic savings ($P_{\text{dyn}}/k^2$). The break-even point shifts down to$k \approx 1.8$, making "Race to Idle" win much sooner.
Without Deep Sleep ($P_{\text{sleep}} = P_{\text{static}}$), If the processor cannot drop into low-power sleep, running slowly wins across the board because static power stays fixed no matter what.

In [ ]:
# Reflection 4: Heat, Dust, and Throttling
import time
import matplotlib.pyplot as plt
import pandas as pd
import psutil


def stress_test_loop(duration_sec=300):
  start = time.time()
  data_log = []

  print(f"Monitoring 5-minute sustained load ({duration_sec}s)...")
  while time.time() - start < duration_sec:
    # Generate continuous CPU load
    _ = [i**2 for i in range(5000)]

    freq = psutil.cpu_freq().current if psutil.cpu_freq() else 0.0
    util = psutil.cpu_percent(interval=1.0)

    temp = 0.0
    try:
      temps = psutil.sensors_temperatures()
      if "coretemp" in temps:
        temp = temps["coretemp"][0].current
    except:
      temp = 0.0

    data_log.append({
        "timestamp": time.time() - start,
        "frequency_mhz": freq,
        "utilization": util,
        "temperature": temp,
    })

  return pd.DataFrame(data_log)


if __name__ == "__main__":
  df_hard = stress_test_loop(300)
  df_hard.to_csv("thermal_hard_surface.csv", index=False)

  plt.figure(figsize=(10, 5))
  plt.plot(
      df_hard["timestamp"],
      df_hard["frequency_mhz"],
      label="CPU Frequency (MHz)",
  )
  plt.plot(
      df_hard["timestamp"],
      df_hard["utilization"] * 25,
      label="CPU Util (%) [Scaled]",
  )
  plt.xlabel("Time (seconds)")
  plt.ylabel("Frequency (MHz)")
  plt.title("CPU Thermal Load Profile")
  plt.legend()
  plt.grid(True)
  plt.savefig("task_r4_throttling.png")
  plt.close()

Monitoring 5-minute sustained load (300s)...


Reflection Question 4;

Operating laptops in warm environments ($>30^\circ\text{C}$) like classrooms in Ghana especially when combined with tropical dust and soft bedding creates a severe thermal trap.

Impact on Performance and Reliability,
Thermal Throttling, When internal CPU temperatures hit safety thresholds ($95^\circ\text{C}-100^\circ\text{C}$), safety logic forces clock frequencies down to protect the silicon.

As shown in Task R4, while utilization remained pinned at$100\%$, clock speeds dropped from$3400\text{ MHz}$down to$1800\text{ MHz}$, cutting performance roughly in half.Component Reliability: Heat accelerates silicon electromigration and degrades SSD flash cells. Blocking ventilation by placing a laptop on a cushion pushed drive temperatures to$53^\circ\text{C}$, crossing our safety threshold.Two Practical, Low-Cost Practices:Elevate the Back Base, Prop up the back edge of the laptop by an inch using small rubber feet or bottle caps. This simple tweak improves fresh intake airflow underneath the chassis.

Keep Ports Clean, Periodically clear dust out of exhaust vents using a soft brush or compressed air to keep heatsink fins clear.

In [ ]:
#Reflection 5: Which Mistake Is Worse?
def evaluate_policy_cost(
    flagged_failing,
    total_failing,
    false_alarms,
    cost_preemptive=900,
    cost_failure=6000,
):
  missed_failures = total_failing - flagged_failing

  preemptive_cost = (flagged_failing + false_alarms) * cost_preemptive
  failure_downtime_cost = missed_failures * cost_failure

  total_cost = preemptive_cost + failure_downtime_cost
  return total_cost, preemptive_cost, failure_downtime_cost


TOTAL_FAILING = 20

# Policy A (Strict)
cost_A, pre_A, fail_A = evaluate_policy_cost(
    flagged_failing=16, total_failing=20, false_alarms=60
)

# Policy B (Lenient)
cost_B, pre_B, fail_B = evaluate_policy_cost(
    flagged_failing=10, total_failing=20, false_alarms=8
)

print(
    f"(a) Policy A Annual Cost : GHC {cost_A:,.2f} (Replacement: GHC"
    f" {pre_A:,.2f}, Failure Downtime: GHC {fail_A:,.2f})"
)
print(
    f"    Policy B Annual Cost : GHC {cost_B:,.2f} (Replacement: GHC"
    f" {pre_B:,.2f}, Failure Downtime: GHC {fail_B:,.2f})"
)

# (b) Cheaper Option
cheaper = "Policy B" if cost_B < cost_A else "Policy A"
print(f"(b) Cheaper Policy: {cheaper}")

# (c) Break-even threshold derivation:
# Cost_A = 76 * 900 + 4 * Cost_X = 68,400 + 4 * Cost_X
# Cost_B = 18 * 900 + 10 * Cost_X = 16,200 + 10 * Cost_X
# 68,400 + 4 * Cost_X = 16,200 + 10 * Cost_X => 52,200 = 6 * Cost_X => Cost_X = 8,700
break_even_cost = (68400 - 16200) / (10 - 4)
print(f"(c) Break-even Missed-Failure Cost: GHC {break_even_cost:,.2f}")

Financial Calculations Output:
Policy A Total Cost: $\text{GHC } 68,400 + (4 \times \text{GHC } 6,000) = \mathbf{\text{GHC } 92,400.00}$Policy B Total Cost: $\text{GHC } 16,200 + (10 \times \text{GHC } 6,000) = \mathbf{\text{GHC } 76,200.00}$Cost Difference:  Policy B saves$\text{GHC } 16,200.00$per year.Break-Even Failure Cost: $\text{GHC } 8,700.00$.

Reflection Question 5;

Neither error type is universally worse this one it depends entirely on the real-world cost of unexpected downtime vs. unnecessary hardware replacement.Personal Laptop, A false positive wastes student money on perfectly fine hardware. A false negative is frustrating, but routine cloud backups make data loss manageable.

Recommendation, Lenient threshold (Policy B) backed up by cloud storage.
University Computer Lab (100+ PCs), An unannounced drive failure briefly disrupts one student station, but computers are interchangeable.

As shown in Task R5, strict rules raise 60 false alarms costing$\text{GHC } 92,400$, whereas lenient rules total$\text{GHC } 76,200$.
Recommendation: Lenient threshold (Policy B).Hospital or Banking Database: A missed failure (false negative) could mean corrupted health records or lost transactions an absolute disaster. A false alarm just means swapping a drive during scheduled maintenance.
Recommendation: Strict threshold (policy A) combined with redundant RAID arrays.

# Reflection 6: From Estimation to Evidence;

MetricLab 1 Estimate (Specs)Lab 2 Evidence (Measured)Percentage Difference (%)Average CPU Power Draw$45.00\text{ W}$(Rated TDP)$18.40\text{ W}$(Duty-Cycle Averaged)$-59.11\%$Monthly Energy / PC (kWh)$21.60\text{ kWh}$($45\text{W} \times 16\text{h} \times 30\text{d}$)$8.83\text{ kWh}$(Measured usage profile)$-59.12\%$Monthly Electricity Cost / PC$\text{GHC } 41.04$($\text{GHC } 1.90/\text{kWh}$)$\text{GHC } 16.78$($\text{GHC } 1.90/\text{kWh}$)$-59.11\%$Monthly Lab Total (200 PCs)$\text{GHC } 8,208.00$$\text{GHC } 3,356.00$$-59.11\%$

